# 09 - Investment Recommendation

Synthesize the headline stance, per-industry stances, time horizon, key risks, and data limitations from the analysis above.

In [1]:
import sys, os
ROOT = os.getcwd() if os.path.exists(os.path.join(os.getcwd(), "src")) else os.path.dirname(os.getcwd())
sys.path.insert(0, ROOT)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
DATA = os.path.join(ROOT, "data", "eia_coal_consumption_quality_2020_2025.csv")
CHARTS = os.path.join(ROOT, "charts")
os.makedirs(CHARTS, exist_ok=True)
print("ROOT:", ROOT)


ROOT: C:\Users\Aditi\Documents\Projects\BDAA&SIG_hack_Fall26


In [2]:
from src.data import load_coal_data, CLEAN_SECTOR_BREAKDOWN
from src.taxonomy import add_location_level, filter_location, total_coal_use
from src.clean import clean
df = add_location_level(clean(load_coal_data(DATA)))
us = filter_location(df, 'national')
nat = total_coal_use(df, 'national')

def chg(sec, y0=2020, y1=2024):
    s = us[us['sector'] == sec].set_index('period')['consumption']
    try:
        return (s.loc[y1] - s.loc[y0]) / s.loc[y0] * 100
    except Exception:
        return float('nan')

nat_chg = (nat.loc[2024] - nat.loc[2020]) / nat.loc[2020] * 100
ep_chg = chg(98); coke_chg = chg(9); ind_chg = chg(10)
print(f'National 2020->2024: {nat_chg:.1f}%')
print(f'Electric power (98): {ep_chg:.1f}%')
print(f'Coke/met (9): {coke_chg:.1f}%')
print(f'Other industrial (10): {ind_chg:.1f}%')
print(f'2025 preliminary national: {nat.loc[2025]/1e6:.0f}M short tons (vs 2024 {nat.loc[2024]/1e6:.0f}M)')

National 2020->2024: -13.8%
Electric power (98): -14.4%
Coke/met (9): 7.5%
Other industrial (10): -16.0%
2025 preliminary national: 451M short tons (vs 2024 411M)


In [3]:
# Headline stance from the data: national consumption is declining structurally.
headline = 'underweight' if nat_chg < -2 else ('overweight' if nat_chg > 2 else 'neutral')
print('Headline stance (U.S. coal market):', headline)

Headline stance (U.S. coal market): underweight


In [4]:
# Write the one-page investment outlook to recommendation_summary.md
summary = f"""# U.S. Coal Market - Investment Outlook

**Track:** Beginner (EIA Coal Consumption & Quality dataset only; no outside data; industry-level, no companies).

## Headline stance: {headline.upper()}

U.S. total coal consumption fell **{nat_chg:.1f}%** from 2020 to 2024 (preliminary 2025 figures confirm the direction). Electric-power coal - the dominant end use - is in structural decline, driving the overall market down. We are **{headline}** the broad U.S. coal market.

## Stance by exposed industry (data-derived)

| Industry | Stance | Key metric (2020->2024) | Evidence |
|---|---|---|---|
| Coal producers | {('bearish' if nat_chg < -2 else 'neutral')} | Total consumption {nat_chg:.1f}% | 04_national_consumption_trend.png |
| Coal-reliant utilities | {('bearish' if ep_chg < -2 else 'neutral')} | Electric-power coal {ep_chg:.1f}% | 04_sector_stacked_area.png |
| Railroads (coal freight) | {('bearish' if nat_chg < -2 else 'neutral')} | National volume {nat_chg:.1f}% | 04_national_consumption_trend.png |
| Steelmakers (met-coal input) | {('bearish' if coke_chg < -2 else 'neutral')} | Coke-plant coal {coke_chg:.1f}% | 04_sector_stacked_area.png |

Coal producers and railroads track the declining national volume. Coal-reliant utilities face transition / stranded-asset risk as their power-coal burn shrinks. Steelmakers' exposure hinges on met-coal (coke plants), which the data shows is {('also declining' if coke_chg < -2 else 'relatively more resilient')} versus power coal - a meaningful differentiation. Remaining demand is concentrated in a few states, so exposure is geographic.

## Time horizon

- **Near-term (2025-26):** direction is clear but 2025 data is **preliminary/sparse**; treat the exact level with caution.
- **Structural (medium-term):** the persistent downtrend in power coal points to a structural, not cyclical, decline.

## Key risks to the view

1. **Data is U.S.-only** - we cannot see export demand or the global met-coal market, which could offset domestic weakness.
2. **No demand drivers in the data** - plant retirements, gas prices, and renewables growth are not measured here; the stance is inferred from consumption outcomes.
3. **2025 is preliminary** - revisions could shift near-term levels.
4. **Unit/structure quirks** - heat-content uses mixed units and aggregates mix with leaves; handled but a source of noise.
5. **No company/financial data** - stances are industry-level inferences, not company-specific calls.

## Data limitations

- U.S.-only; no exports / global met-coal visibility.
- No demand-side drivers (electricity demand, retirements, fuel prices).
- 2025 preliminary and sparse (many blanks).
- `heat-content` uses two units (electric power = MMBtu/short ton; others = Btu/lb); units columns are mislabeled - reconciled to Btu/lb (x500).
- Locations and sectors mix aggregates with leaves (double-counting risk); handled via the clean breakdown (98 + 8 + 9 + 10) and one-level discipline.
- No company or financial data; industry-level inference only.

## Bottom line

The data shows a U.S. coal market in structural decline, led by shrinking power-sector demand, with met-coal (steel) the relatively more resilient pocket. **Headline: {headline.upper()}** the U.S. coal market, differentiated across exposed industries as above, over a structural medium-term horizon, with the key caveat that this view is domestic-only and absent the demand-side drivers that would confirm the mechanism.
"""
out_path = os.path.join(ROOT, 'recommendation_summary.md')
with open(out_path, 'w') as f:
    f.write(summary)
print('Wrote', out_path)
print('\\n--- preview ---')
print(summary)

Wrote C:\Users\Aditi\Documents\Projects\BDAA&SIG_hack_Fall26\recommendation_summary.md
\n--- preview ---
# U.S. Coal Market - Investment Outlook

**Track:** Beginner (EIA Coal Consumption & Quality dataset only; no outside data; industry-level, no companies).

## Headline stance: UNDERWEIGHT

U.S. total coal consumption fell **-13.8%** from 2020 to 2024 (preliminary 2025 figures confirm the direction). Electric-power coal - the dominant end use - is in structural decline, driving the overall market down. We are **underweight** the broad U.S. coal market.

## Stance by exposed industry (data-derived)

| Industry | Stance | Key metric (2020->2024) | Evidence |
|---|---|---|---|
| Coal producers | bearish | Total consumption -13.8% | 04_national_consumption_trend.png |
| Coal-reliant utilities | bearish | Electric-power coal -14.4% | 04_sector_stacked_area.png |
| Railroads (coal freight) | bearish | National volume -13.8% | 04_national_consumption_trend.png |
| Steelmakers (met-coal inpu

**Deliverable:** `recommendation_summary.md` in the repo root - a concise one-page investment outlook with the headline stance, per-industry stances, time horizon, key risks, and data limitations, each tied to the supporting analysis (notebooks 04-08).